# torch-np-classifier — Quick Start

Minimal notebook covering:
1. **Predictions** — single molecule and batch
2. **Molecular embeddings** — penultimate-layer features
3. **Explainability** — SHAP-based fingerprint highlighting

Pretrained checkpoints (~300 MB) are downloaded automatically on first prediction and cached for the rest of the session.

## 0. Install

In [ ]:
%pip install -q torch-np-classifier
%pip install -q shap matplotlib Pillow

---
## 1. Predictions

`NPClassifierPipeline()` loads pretrained models automatically on first call.

In [ ]:
from torch_np_classifier import NPClassifierPipeline

pipeline = NPClassifierPipeline()

# Single molecule
result = pipeline.predict("O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12")  # quercetin
print("Quercetin")
print("  pathway    :", result["pathway"])
print("  superclass :", result["superclass"])
print("  class      :", result["class"])
print("  glycoside  :", result["isglycoside"])

In [ ]:
import pandas as pd

molecules = {
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "morphine": "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@@H]3[C@@H]1C5",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
}

results = pipeline.predict(list(molecules.values()))

pd.DataFrame(
    [
        {
            "molecule": name,
            "pathway": ", ".join(r["pathway"]) or "—",
            "superclass": ", ".join(r["superclass"]) or "—",
            "class": ", ".join(r["class"]) or "—",
        }
        for name, r in zip(molecules, results)
    ]
).set_index("molecule")

---
## 2. Molecular embeddings

`predict_embeddings` returns penultimate-layer activations as a `(N, 1536)` float32 array.  
The `level` argument selects which model to extract from: `"pathway"`, `"superclass"`, or `"class"`.

In [ ]:
smiles_list = list(molecules.values())

embeddings = pipeline.predict_embeddings(smiles_list, level="class")
print("Shape :", embeddings.shape)  # (4, 1536)
print("dtype :", embeddings.dtype)

---
## 3. Explainability

`explain()` computes SHAP values with `shap.GradientExplainer` and returns a figure showing:
- the molecule with the most important fingerprint bits highlighted
- a SHAP bar chart per predicted level
- a grid of the corresponding Morgan-environment fragments

A bundled 200-molecule background is used by default — no extra setup needed.

In [ ]:
import matplotlib.pyplot as plt

fig = pipeline.explain("CC(=O)Oc1ccccc1C(=O)O", k=6)  # aspirin
plt.show()

In [ ]:
# Single-level explanation (one panel per predicted label at that level)
fig = pipeline.explain_bits(
    "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",  # quercetin
    level="class",
    k=6,
)
plt.show()